# Persian speech-to-text: model comparison

Five candidate models are evaluated on the project's own recordings (112 samples with reference transcripts). The metric is corpus-level word error rate (WER) after text normalization; see the note on `accuracy_percent` in the results section.

| # | Model | Type |
|---|---|---|
| 1 | `openai/whisper-large-v3` | Whisper, multilingual |
| 2 | `nvidia/stt_fa_fastconformer_hybrid_large` | FastConformer (NeMo), Persian |
| 3 | `jonatasgrosman/wav2vec2-large-xlsr-53-persian` | Wav2Vec 2.0 + CTC, Persian fine-tune |
| 4 | `m3hrdadfi/wav2vec2-large-xlsr-persian-v3` | Wav2Vec 2.0 + CTC, Persian fine-tune |
| 5 | `vhdm/whisper-large-fa-v1` | Whisper large, Persian fine-tune |

**Note on model 5.** An earlier candidate, `facebook/wav2vec2-large-xlsr-53`, was dropped: it is a self-supervised pretrained checkpoint without a fine-tuned CTC head, so it cannot produce meaningful transcripts. `vhdm/whisper-large-fa-v1` was used instead; because it shares its architecture with model 1, the pair shows the effect of a dedicated Persian fine-tune against the general multilingual model.

**Input.** The notebook expects a dataset root containing an `audio/` folder and a `metadata.xlsx` sheet with `file_name` and `transcript` columns. The recordings are private and are not part of this repository.

## 1. Setup and data loading

In [1]:
# NeMo pulls in its own numba; the preinstalled numba-cuda conflicts with it on Kaggle.
!pip install -q -U "numpy<2.2" scipy transformers accelerate jiwer soundfile librosa hazm "nemo_toolkit[asr]"
!pip uninstall -y numba-cuda -q


In [2]:
import os
import re
import gc
import time
import tempfile

import torch
import numpy as np
import pandas as pd
import librosa
import soundfile as sf

from jiwer import wer
from hazm import Normalizer

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32
print("Device:", device)

TARGET_SR = 16000
normalizer = Normalizer()

# ---- configuration ----
# Dataset layout:
#   <DATASET_ROOT>/audio/          audio files
#   <DATASET_ROOT>/metadata.xlsx   Sheet1 with columns file_name, emotion, transcript
DATASET_ROOT = "/kaggle/input/datasets/ftmjfr/audio-transcript/Audio_transcript_dataset/voice_dataset"
AUDIO_DIR = os.path.join(DATASET_ROOT, "audio")
METADATA_XLSX = os.path.join(DATASET_ROOT, "metadata.xlsx")
METADATA_SHEET = "Sheet1"
N_PRINT_SAMPLES = 20  # number of (prediction, reference) pairs printed per model
# -------------------

import glob

meta_df = pd.read_excel(METADATA_XLSX, sheet_name=METADATA_SHEET)
print(f"Loaded metadata with {len(meta_df)} rows.")

# file_name in the sheet has no extension (e.g. "001_HAP"), so we accept
# any common audio extension found in the audio folder.
AUDIO_EXTS = [".wav", ".mp3", ".flac", ".ogg", ".m4a"]


def find_audio_path(file_name: str):
    base = str(file_name).strip()
    # in case the column already includes an extension
    direct = os.path.join(AUDIO_DIR, base)
    if os.path.exists(direct):
        return direct
    for ext in AUDIO_EXTS:
        candidate = os.path.join(AUDIO_DIR, base + ext)
        if os.path.exists(candidate):
            return candidate
    # glob fallback (for possible subfolders inside audio/)
    matches = glob.glob(os.path.join(AUDIO_DIR, "**", base + ".*"), recursive=True)
    return matches[0] if matches else None


manifest_rows = []
missing = []
for _, row in meta_df.iterrows():
    audio_path = find_audio_path(row["file_name"])
    if audio_path is None:
        missing.append(row["file_name"])
        continue
    manifest_rows.append({"audio_path": audio_path, "transcript": row["transcript"]})

if missing:
    print(f"[warn] {len(missing)} audio file(s) not found (sample): {missing[:5]}")

manifest_df = pd.DataFrame(manifest_rows)
print(f"Manifest ready with {len(manifest_df)} matched samples.")


def load_audio_robust(path: str, sr: int = TARGET_SR):
    """Try librosa/soundfile first; if the format (e.g. m4a) isn't
    supported, convert to a temporary wav with ffmpeg and read that."""
    try:
        array, _ = librosa.load(path, sr=sr, mono=True)
        return array
    except Exception:
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            tmp_wav = tmp.name
        try:
            result = os.system(
                f'ffmpeg -y -loglevel error -i "{path}" -ar {sr} -ac 1 "{tmp_wav}"'
            )
            if result != 0:
                raise RuntimeError(f"ffmpeg failed to convert {path}")
            array, _ = librosa.load(tmp_wav, sr=sr, mono=True)
            return array
        finally:
            if os.path.exists(tmp_wav):
                os.remove(tmp_wav)


samples = []
failed_audio = []
for _, row in manifest_df.iterrows():
    try:
        array = load_audio_robust(row["audio_path"], sr=TARGET_SR)
    except Exception as e:
        failed_audio.append((row["audio_path"], str(e)))
        continue
    samples.append({"audio_path": row["audio_path"], "array": array, "sr": TARGET_SR,
                     "reference": str(row["transcript"])})

if failed_audio:
    print(f"[warn] {len(failed_audio)} audio file(s) could not be read (sample): {failed_audio[:3]}")
print(f"Loaded {len(samples)} audio samples.")


Device: cuda
Loaded metadata with 112 rows.
Manifest ready with 112 matched samples.
Loaded 112 audio samples.


## 2. Text normalization and WER

References and hypotheses go through the same normalization (hazm normalizer, diacritics and punctuation removed) before WER is computed. Pairs where either side is empty after cleaning are skipped and reported as `valid_samples`.

In [3]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = normalizer.normalize(text)
    text = re.sub(r"[\u064B-\u065F\u0670\u06D6-\u06ED]", "", text)  # remove diacritics
    text = re.sub(r"[^\w\s]", " ", text)  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()
    return text


def compute_wer(references, hypotheses):
    refs_clean = [clean_text(r) for r in references]
    hyps_clean = [clean_text(h) for h in hypotheses]
    pairs = [(r, h) for r, h in zip(refs_clean, hyps_clean) if len(r) > 0 and len(h) > 0]
    if not pairs:
        return 1.0, 0
    refs_valid = [p[0] for p in pairs]
    hyps_valid = [p[1] for p in pairs]
    return wer(refs_valid, hyps_valid), len(pairs)


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 3. Inference wrappers

One function per model family: Whisper (transformers pipeline), Wav2Vec 2.0 CTC (greedy decoding), and NeMo FastConformer. Each returns one hypothesis per sample and frees GPU memory afterwards.

In [4]:
def transcribe_all_whisper(model_id, samples):
    from transformers import WhisperProcessor, WhisperForConditionalGeneration, pipeline
    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True
    ).to(device)
    asr = pipeline(
        task="automatic-speech-recognition",
        model=model, tokenizer=processor.tokenizer, feature_extractor=processor.feature_extractor,
        torch_dtype=torch_dtype, device=0 if device == "cuda" else -1,
        generate_kwargs={"language": "fa", "task": "transcribe"}, chunk_length_s=30,
    )
    hyps = []
    for s in samples:
        try:
            hyps.append(asr(s["array"])["text"])
        except Exception as e:
            print(f"  [warn] Whisper failed on {s['audio_path']}: {e}")
            hyps.append("")
    del model, asr
    free_gpu()
    return hyps


def transcribe_all_wav2vec2_ctc(model_id, samples):
    from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
    processor = Wav2Vec2Processor.from_pretrained(model_id)
    model = Wav2Vec2ForCTC.from_pretrained(model_id).to(device)
    model.eval()
    hyps = []
    for s in samples:
        try:
            inputs = processor(s["array"], sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
            input_values = inputs.input_values.to(device)
            with torch.no_grad():
                logits = model(input_values).logits
            pred_ids = torch.argmax(logits, dim=-1)
            text = processor.batch_decode(pred_ids)[0]
            hyps.append(text)
        except Exception as e:
            print(f"  [warn] Wav2Vec2 ({model_id}) failed on {s['audio_path']}: {e}")
            hyps.append("")
    del model, processor
    free_gpu()
    return hyps


def transcribe_all_nemo(model_id, samples):
    import nemo.collections.asr as nemo_asr
    model = nemo_asr.models.ASRModel.from_pretrained(model_name=model_id).to(device)
    model.eval()
    hyps = []
    for s in samples:
        tmp_path = None
        try:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp_path = tmp.name
            sf.write(tmp_path, s["array"], s["sr"])
            result = model.transcribe([tmp_path], verbose=False)[0]
            text = result.text if hasattr(result, "text") else result
            hyps.append(text or "")
        except Exception as e:
            print(f"  [warn] NeMo failed on {s['audio_path']}: {e}")
            hyps.append("")
        finally:
            if tmp_path and os.path.exists(tmp_path):
                os.remove(tmp_path)
    del model
    free_gpu()
    return hyps


## 4. Evaluation helper

In [ ]:
references = [s["reference"] for s in samples]
summary_rows = []
all_hypotheses = {}  # model name -> list of predictions (for further inspection if needed)

def run_model(model_id, framework, note=None):
    print("\n" + "=" * 70)
    print(f"Model: {model_id}")
    if note:
        print(f"Note: {note}")
    print("=" * 70)
    start = time.time()
    try:
        hyps = TRANSCRIBE_FN[framework](model_id, samples)
    except Exception as e:
        print(f"[error] {model_id} failed and was skipped: {e}")
        return
    elapsed = time.time() - start
    all_hypotheses[model_id] = hyps

    if N_PRINT_SAMPLES > 0:
        print(f"\n--- first {min(N_PRINT_SAMPLES, len(samples))} samples (prediction / reference) ---")
        for i in range(min(N_PRINT_SAMPLES, len(samples))):
            print(f"[{i+1}] pred: {hyps[i]}")
            print(f"     ref:  {references[i]}")

    model_wer, n_valid = compute_wer(references, hyps)
    accuracy = (1 - model_wer) * 100  # complement of WER, not a classification accuracy
    summary_rows.append({
        "model": model_id,
        "wer_percent": round(model_wer * 100, 2),
        "accuracy_percent": round(accuracy, 2),
        "valid_samples": n_valid,
        "total_samples": len(samples),
        "time_seconds": round(elapsed, 1),
    })
    print(f"\n>>> {model_id}: WER={model_wer*100:.2f}%  Accuracy={accuracy:.2f}%  "
          f"(n={n_valid}/{len(samples)}, {elapsed:.1f}s)")


## 5. Run all models

In [ ]:
MODELS = [
    {"name": "openai/whisper-large-v3",                       "framework": "whisper"},
    {"name": "nvidia/stt_fa_fastconformer_hybrid_large",       "framework": "nemo"},
    {"name": "jonatasgrosman/wav2vec2-large-xlsr-53-persian",  "framework": "wav2vec2"},
    {"name": "m3hrdadfi/wav2vec2-large-xlsr-persian-v3",       "framework": "wav2vec2"},
    {"name": "vhdm/whisper-large-fa-v1",                       "framework": "whisper"},
]

TRANSCRIBE_FN = {
    "whisper": transcribe_all_whisper,
    "wav2vec2": transcribe_all_wav2vec2_ctc,
    "nemo": transcribe_all_nemo,
}


In [ ]:
run_model("openai/whisper-large-v3", "whisper")



Model: openai/whisper-large-v3


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step


--- first 20 samples (prediction / reference) ---
[1] pred:  به نخر بوس روش جوای چقدر خوب قولم خیلی زیاد شد و ماشالله
     ref:  بالاخره بورس رشد کرد وای چقدر خوب پولام خیلی زیاد شد ماشالا
[2] pred:  امروز من دیگه حسن هیچ کاری ندارم زنم مرده نمیدونم چی کار کنم خواهم یک زن دیگه بسونم
     ref:  امروز من دیگه حوصله هیچ کاری رو ندارم زنم مرده نمیدونم چیکار کنم میخوام برم یه زن دیگه بسونم
[3] pred:  رفتم خونه ما در شهر، چقدر به امتیکه انداخت، کاش جوابش داده بودم. عصامم رو خورد کرد.
     ref:  رفتم خونه مادرشوهر چقدر بهم تیکه انداخت کاش جوابش رو داده بودم اعصابم رو خورد کرد
[4] pred:  میایی با هم بریم بیرون؟
     ref:  میای باهم بریم بیرون
[5] pred:  ای کاش میشد برگشت به اون روزا اون روزا که من تو آج خودم بودم
     ref:  ای کاش میشد برگشت به اون روزا اون روزا که من تو اوج خودم بودم
[6] pred:  امروز فهمیدم اینکه دوستام تصادف کرده و بیمارستانه و خیلی نگردنشم.
     ref:  امروز فهمیدم یکی از دوستام تصادف کرده و بیمارستانه و خیلی نگرانشم
[7] pred:  امروز به لحظه جشن فارغا تحصیلیمون رو گرفتیم
  

In [ ]:
run_model("nvidia/stt_fa_fastconformer_hybrid_large", "nemo")



Model: nvidia/stt_fa_fastconformer_hybrid_large


stt_fa_fastconformer_hybrid_large.nemo:   0%|          | 0.00/459M [00:00<?, ?B/s]

[NeMo I 2026-09-28 06:55:43 mixins:194] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-09-28 06:55:44 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: dummy
    sample_rate: 16000
    batch_size: 1
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 10
    min_duration: 0.5
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    use_lhotse: true
    lhotse:
      shar_path: /data_artifacts/data/shar/train
      batch_duration: 1200
      quadratic_duration: 15
      num_buckets: 10
      num_cuts_for_bins_estimate: 10000
      buffer_size: 10000
      shuffle_buffer_size: 10000
    
[NeMo W 2026-09-28 06:55:44 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a vali

[NeMo I 2026-09-28 06:55:45 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-09-28 06:55:45 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-09-28 06:55:45 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-09-28 06:55:46 save_restore_connector:287] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--stt_fa_fastconformer_hybrid_large/snapshots/249cf5bf70dda7220a60ddeeecff2f6aad8e1784/stt_fa_fastconformer_hybrid_large.nemo.


[NeMo W 2026-09-28 06:55:46 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-09-28 06:55:46 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
[NeMo W 2026-09-28 06:55:46 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-09-28 06:55:46 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is


--- first 20 samples (prediction / reference) ---
[1] pred: بالاخره بوس جواب چقدر خوب خیلی زیاد شاهد
     ref:  بالاخره بورس رشد کرد وای چقدر خوب پولام خیلی زیاد شد ماشالا
[2] pred: امروز من یه حوصله کار ندارم نمونه
     ref:  امروز من دیگه حوصله هیچ کاری رو ندارم زنم مرده نمیدونم چیکار کنم میخوام برم یه زن دیگه بسونم
[3] pred: رفتم خونه مادرشوهرچم اسباب خرد کرد
     ref:  رفتم خونه مادرشوهر چقدر بهم تیکه انداخت کاش جوابش رو داده بودم اعصابم رو خورد کرد
[4] pred: میای با هم بریم بیرون
     ref:  میای باهم بریم بیرون
[5] pred: ای کاش میشد برگشتن تو اوج خودم بودن
     ref:  ای کاش میشد برگشت به اون روزا اون روزا که من تو اوج خودم بودم
[6] pred: امروز فهمیدم اینکه دوستام تصادف کرده و بیمارستان خیلی نگران
     ref:  امروز فهمیدم یکی از دوستام تصادف کرده و بیمارستانه و خیلی نگرانشم
[7] pred: امروز بالاخره جشن فقط فصل گرفتیم
     ref:  امروز بالاخره جشن فارغ التحصیلیمون رو گرفتیم 
[8] pred: وای فردا امتحان داریم خیلی هم درس و هیچی نخوندم
     ref:  وای فردا امتحان داریم خیلی هم درس سختیه و 

In [ ]:
run_model("jonatasgrosman/wav2vec2-large-xlsr-53-persian", "wav2vec2")



Model: jonatasgrosman/wav2vec2-large-xlsr-53-persian


preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]


--- first 20 samples (prediction / reference) ---
[1] pred: باناخره بوس رشت گ واقای چقدر خوب گلاب خیلی زیا اشو ماشال ز
     ref:  بالاخره بورس رشد کرد وای چقدر خوب پولام خیلی زیاد شد ماشالا
[2] pred: امروز مند یه حوصر هیچکاریه ندارم زنم مردهنیمیدم چیکار کنه میخابارم یه زن دیگه بسونم
     ref:  امروز من دیگه حوصله هیچ کاری رو ندارم زنم مرده نمیدونم چیکار کنم میخوام برم یه زن دیگه بسونم
[3] pred: رفتم خونه مادر شاید چقدِ به امتی که انداخت کاش ت قابش داده بودم قصلممو خورد کرد
     ref:  رفتم خونه مادرشوهر چقدر بهم تیکه انداخت کاش جوابش رو داده بودم اعصابم رو خورد کرد
[4] pred: میای با هم بریم بیرون
     ref:  میای باهم بریم بیرون
[5] pred: اهکاش می شد برگش به اون روزا اونزا که من تو ارجه خودم بودم
     ref:  ای کاش میشد برگشت به اون روزا اون روزا که من تو اوج خودم بودم
[6] pred: امروز فهمیدم اینکه دوستام تصادف کرده و بیمارستانهم و خیلی نگرانشم
     ref:  امروز فهمیدم یکی از دوستام تصادف کرده و بیمارستانه و خیلی نگرانشم
[7] pred: امروز بالاخره جش  فاریغ ا تحصیلیمونو گرفتیم
     ref:  امرو

In [ ]:
run_model("m3hrdadfi/wav2vec2-large-xlsr-persian-v3", "wav2vec2")



Model: m3hrdadfi/wav2vec2-large-xlsr-persian-v3


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/307 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/399 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]


--- first 20 samples (prediction / reference) ---
[1] pred: بالاخره بوست رشد جه وقای طقدر خوم کلام خیلی زاجو ماشنت
     ref:  بالاخره بورس رشد کرد وای چقدر خوب پولام خیلی زیاد شد ماشالا
[2] pred: امروزمن ی حوصل هیچکاری ندارم زنم مرده نمی ندم چیکار کن مخا برم یه زن دیگه بسونم
     ref:  امروز من دیگه حوصله هیچ کاری رو ندارم زنم مرده نمیدونم چیکار کنم میخوام برم یه زن دیگه بسونم
[3] pred: رفتم خون مادر شوهر چقدر بمتیکه انداخت کاش تقابشو داده بودم اا ممو خورد کرد
     ref:  رفتم خونه مادرشوهر چقدر بهم تیکه انداخت کاش جوابش رو داده بودم اعصابم رو خورد کرد
[4] pred: میای با هم بریم بیرون
     ref:  میای باهم بریم بیرون
[5] pred: ای کاش می شد برگشت بیه اون روزا اوزا که من تو اوج خودم بودم
     ref:  ای کاش میشد برگشت به اون روزا اون روزا که من تو اوج خودم بودم
[6] pred: امروز فهمیدم اینکه دوستام تصادف کرده و بیمارستانه و خیلی نگرانشه
     ref:  امروز فهمیدم یکی از دوستام تصادف کرده و بیمارستانه و خیلی نگرانشم
[7] pred: امروز بالاخره جشن فارغ ال تحصیلیمونو گرفتیم
     ref:  امروز بالاخره جشن

In [ ]:
run_model("vhdm/whisper-large-fa-v1", "whisper")



Model: vhdm/whisper-large-fa-v1


preprocessor_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).



--- first 20 samples (prediction / reference) ---
[1] pred: به در هر بوس روش جاید، وای چقدر خوب، پولم خیلی زیاد چاید، ماشاالله
     ref:  بالاخره بورس رشد کرد وای چقدر خوب پولام خیلی زیاد شد ماشالا
[2] pred: امروز من دیگه حوصل هیچ کاریه ندارم، زنم مرده، نمی‌بیندم چی کار کنم، می‌خواهم یه زندگی بسونم.
     ref:  امروز من دیگه حوصله هیچ کاری رو ندارم زنم مرده نمیدونم چیکار کنم میخوام برم یه زن دیگه بسونم
[3] pred: رفتم خون مادرشا، چقدر به امتیک انداخت کاش تو آبشه داده بودم، از سامم خورد کرد.
     ref:  رفتم خونه مادرشوهر چقدر بهم تیکه انداخت کاش جوابش رو داده بودم اعصابم رو خورد کرد
[4] pred: بیایی با هم بریم بیرون؟
     ref:  میای باهم بریم بیرون
[5] pred: ای کاش می‌شد برگش به آن روزا، آن روزا که من تو عوج خودم بودم.
     ref:  ای کاش میشد برگشت به اون روزا اون روزا که من تو اوج خودم بودم
[6] pred: امروز فهمیدم این که دوستان تصادف کرده و بیمارستانه و خیلی نگرانشم
     ref:  امروز فهمیدم یکی از دوستام تصادف کرده و بیمارستانه و خیلی نگرانشم
[7] pred: امروز به لخه‌ی جشن فارغ‌التحصیلی اونو 

## 6. Results

`accuracy_percent` is `100 - WER`, reported only for readability. It is not a classification accuracy and can be negative when a model inserts many extra words.

In [ ]:
final_df = (
    pd.DataFrame(summary_rows)
    .drop_duplicates(subset="model", keep="first")
    .sort_values("wer_percent")
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("Final model comparison table (sorted by WER, lower is better)")
print("=" * 70)
print(final_df.to_string(index=False))

os.makedirs("/kaggle/working", exist_ok=True)
final_df.to_csv("/kaggle/working/stt_model_comparison_5models.csv", index=False)
print("\nSaved comparison table to /kaggle/working/stt_model_comparison_5models.csv")



Final model comparison table (sorted by WER, lower is better)
                                        model  wer_percent  accuracy_percent  valid_samples  total_samples  time_seconds
                      openai/whisper-large-v3        35.33             64.67            112            112         285.9
     m3hrdadfi/wav2vec2-large-xlsr-persian-v3        37.51             62.49            112            112          27.3
jonatasgrosman/wav2vec2-large-xlsr-53-persian        40.46             59.54            112            112          33.8
                     vhdm/whisper-large-fa-v1        41.50             58.50            112            112          91.2
     nvidia/stt_fa_fastconformer_hybrid_large        43.97             56.03            112            112          29.7

Saved comparison table to /kaggle/working/stt_model_comparison_5models.csv
